In [ ]:
# =====================================================================
# PLANTILLA GENÉRICA: Clasificación Multiclase con Imágenes de Drive
# =====================================================================

# ---------------------------------------------------------------------
# PASO 1: Conectar a Google Drive
# ---------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# ---------------------------------------------------------------------
# PASO 2: Importar Librerías
# ---------------------------------------------------------------------
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Selección automática de hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Trabajando con el dispositivo: {device}\n")

# ---------------------------------------------------------------------
# PASO 3: Configurar Rutas y Número de Clases (¡MODIFICA AQUÍ!)
# ---------------------------------------------------------------------
# Ajusta el nombre de la carpeta raíz si es necesario
RUTA_TRAIN = '/content/drive/MyDrive/CNN_Multiclase/TRAIN/'
RUTA_TEST = '/content/drive/MyDrive/CNN_Multiclase/TEST/'

# ---------------------------------------------------------------------
# PASO 4: Carga de Datos y Auto-detección de Clases
# ---------------------------------------------------------------------
transformaciones = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageFolder(root=RUTA_TRAIN, transform=transformaciones)
test_dataset = ImageFolder(root=RUTA_TEST, transform=transformaciones)

# Guardamos dinámicamente cuántas clases detectó ImageFolder en Drive
NUM_CLASES = len(train_dataset.classes)

print(f"¡Configuración Multiclase Detectada!")
print(f"Número de clases encontradas: {NUM_CLASES}")
print(f"Nombres de las clases: {train_dataset.classes}")
print(f"Mapeo de índices: {train_dataset.class_to_idx}\n")

BATCH_SIZE = 32
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ---------------------------------------------------------------------
# PASO 5: Arquitectura de la CNN Adaptable a N Clases
# ---------------------------------------------------------------------
class CNNMulticlass(nn.Module):
    def __init__(self, num_clases_salida):
        super(CNNMulticlass, self).__init__()

        # Extractor de características (Igual al modelo base)
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 112x112

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 56x56
        )

        # Clasificador
        self.classifier = nn.Sequential(
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            # ¡CLAVE!: En lugar de un número fijo (2), recibe la variable dinámica de clases
            nn.Linear(128, num_clases_salida)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x

# Pasamos NUM_CLASES como argumento al constructor
model = CNNMulticlass(num_clases_salida=NUM_CLASES).to(device)

# ---------------------------------------------------------------------
# PASO 6: Función de Pérdida y Optimidad
# ---------------------------------------------------------------------
# Nota para la clase: CrossEntropyLoss maneja internamente Softmax, por lo que
# es perfectamente compatible tanto para biclase como para multiclase en PyTorch.
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# ---------------------------------------------------------------------
# PASO 7: Bucle de Entrenamiento
# ---------------------------------------------------------------------
NUM_EPOCHS = 5
print("Iniciando el entrenamiento multiclase...")

for epoch in range(NUM_EPOCHS):
    model.train()
    loss_acumulada = 0.0

    for imagenes, etiquetas in train_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)

        predicciones = model(imagenes)
        loss = criterion(predicciones, etiquetas)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_acumulada += loss.item()

    print(f"Época [{epoch+1}/{NUM_EPOCHS}] - Pérdida Promedio: {loss_acumulada/len(train_loader):.4f}")

print("¡Entrenamiento concluido!\n")

# ---------------------------------------------------------------------
# PASO 8: Evaluación Final en el Set de Prueba (Test)
# ---------------------------------------------------------------------
model.eval()
correctos = 0
total_imagenes = 0

with torch.no_grad():
    for imagenes, etiquetas in test_loader:
        imagenes, etiquetas = imagenes.to(device), etiquetas.to(device)
        outputs = model(imagenes)
        _, predicciones_indices = torch.max(outputs.data, 1)

        total_imagenes += etiquetas.size(0)
        correctos += (predicciones_indices == etiquetas).sum().item()

accuracy = 100 * correctos / total_imagenes
print(f"Exactitud (Accuracy) final en TEST para las {NUM_CLASES} clases: {accuracy:.2f}%")

Generar imagen de como funciona YOLO paso a paso